## The objective of this notebook is to extract the sentiment from the ranked document

In [50]:
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
import pandas as pd
import numpy as np
import re
#from unidecode import unidecode


<b> Retrive the company name so that the correct file can be read

In [51]:
%store -r ipo_company_name
#%run ./IPO_InformationRetrieval-v0.1.ipynb

In [52]:
#ipo_company_name = "Sona Comstar"

company_name = ipo_company_name + '_IPO'
dir_name=ipo_company_name

file_name = dir_name+"/"+company_name+'_RankedDocuments.csv'
print("file name : ", file_name)

file name :  Behari Lal Engineering/Behari Lal Engineering_IPO_RankedDocuments.csv


<b>Read the file that has the ranked text/document </b>

In [53]:

data=pd.read_csv(file_name,sep='\t')
data.head()
data.drop(data.columns[0],axis=1,inplace=True)
data.head()
#data = data.dropna()
#data = data.reset_index(drop=True)

,docid,title_snippet
0,G1,behari lal engineering ipo opens aug price rs
1,G2,behari lal engineering fixes price band at rs
2,G3,ipo gmps dhoot transmission molbio diagnostics...
3,G4,behari lal engineering sets rs price band for ...
4,G5,behari lal engineering ltd ipo


<b> Lemmatize the words in the document

In [54]:
def lemmatize_words(words):

    lemmatized_words = [WordNetLemmatizer().lemmatize(word, 'v') for word in words]
    
    return lemmatized_words
word_pattern = re.compile('\w+')



<b> Set the stopwords to 'English' and lemmatize them

In [55]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/aditya/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [56]:
from string import punctuation 
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize


nltk.download('stopwords')
nltk.download('punkt')
stopwords = set(stopwords.words('english'))

#stopwords

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/aditya/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /Users/aditya/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


<b> This function processes the text by removing any URLs

In [57]:
print("ipo company name : ", ipo_company_name.lower())

ipo company name :  behari lal engineering


In [58]:
# a_string = "abc [!? 123"
# alphanumeric = ""

# for character in a_string:
#     if character.isalnum():
#         alphanumeric += character

# print(alphanumeric)

In [59]:
def processText(text):
    text1 = ""
    text2 = ""
    text3 = ""
    text = text.lower ()
    text = re.sub('((www\.[^\s]+)|(https?://[^\s]+))', ' ', text) # remove URLs
    text = re.sub('\[\]',' ', text)
    #text = re.sub('@[^\s]+', ' ', text) # remove usernames (i.e. words starting with @)
    #text = re.sub(r'#([^\s]+)', r' ', text) # remove the # in #hashtag   
    #text = text.replace(ipo_company_name.lower(),"")
    text = text.replace("'","")
    text = re.sub('[^A-Za-z0-9]+', ' ', text)
    
    #text = word_tokenize(text)
    
    #return [word for word in text if word not in stopwords]
    return text

    

In [60]:
#data.title_snippet = data.title_snippet.apply(lambda x: processText(x, unidecode))

<b> This function removes the stopwords in the text

In [61]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/aditya/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [62]:
data = data.dropna()
data = data.reset_index(drop=True)

processed=[]
distance=[]
#print(data)

for text in data['title_snippet']:
    #print("text :", text)   
   
    cleaned = processText(text)
    word_tokens = word_tokenize(cleaned) 
    
    filtered_sentence = [word for word in word_tokens if word not in stopwords]
    filtered_sentence = [] 
  
    for w in word_tokens: 
        if w not in stopwords: 
            filtered_sentence.append(w) 
    
    processed.append(filtered_sentence)
    

<b> Write the text to a file after the stop words are removed

In [63]:
data["processed"] = processed
data.iloc[:, 2:]

processed_file = dir_name+"/"+company_name

data["processed"].to_csv(processed_file+"_processed.txt", index=False,header=None)

<b> Vectorize the words in the text (after processing). <br>
    The sentences are used instead of words to vectorize

<!-- from gensim.models import Word2Vec

model = Word2Vec(sentences=processed, vector_size=100, window=4, min_count=1, workers=4)
model.train(processed, total_examples=1, epochs=1) -->

## TFIDFVectorizer

In [64]:
from sklearn.feature_extraction.text import TfidfVectorizer, TfidfTransformer

<b> The words have some undesired characters and the below function cleans the text

In [65]:
def clean_text(text):
    text = text.replace("'","")
    text = text.replace(",","")
    text = text.replace("[","")
    text = text.replace("]","")
    text = text.replace("\"","")
    return text

<!-- print('\n \n Scores')
scores = pd.DataFrame(data=[test_scores])
scores.columns = ['accuracy']
scores = scores.T
scores.columns = ['scores']
display(scores) -->

In [66]:
import pysentiment2 as ps
import pandas as pd

# Runtime patch to fix category dtype mismatch in pysentiment2 with modern pandas
def patched_init_dict(self):
    data = pd.read_csv(self.PATH, dtype=str)
    self._posset = set(data.query('Positiv == "Positiv"')['Entry'].apply(self.tokenize_first).dropna())
    self._negset = set(data.query('Negativ == "Negativ"')['Entry'].apply(self.tokenize_first).dropna())

ps.HIV4.init_dict = patched_init_dict


In [67]:
hiv4 = ps.HIV4()

In [68]:
df = pd.read_csv(processed_file+"_processed.txt", header=None)

In [69]:
polarity = 0
positive = 0
negative = 0
for each in data['title_snippet']:
    tokens = hiv4.tokenize(each)
    score = hiv4.get_score(tokens)
   
    polarity = polarity+score["Polarity"]
    positive = positive + score["Positive"]
    negative = negative + score["Negative"]

In [70]:
polarity , positive, negative

(np.float64(14.999986000013505), np.int64(17), np.int64(0))

In [71]:
row = ['latest', 'news', 'sona', 'comstar', 'gets']
len(row)


5

In [72]:
token_list = []
for each in data['title_snippet']:
    token_list.append(word_tokenize(each))

#print("token_list : ",token_list)

In [73]:
df["words"] = df

In [74]:
total_word_list=[]
for row in df[0]:
    word_list = row.split()
    #print( "word list : ", word_list, " length : ",len(word_list))
    for i in range(len(word_list)):
        #print("word_list : ",clean_text(word_list[i]))
        total_word_list.append(clean_text(word_list[i]))

In [75]:
len(total_word_list)

194

In [76]:
lm = ps.LM()

In [77]:
polarity = 0
positive = 0
negative = 0
for each in data['title_snippet']:
    #print("each : ",each)
    tokens = lm.tokenize(each)
    score = lm.get_score(tokens)
   
    polarity = polarity+  score["Polarity"]
    positive = positive + score["Positive"]
    negative = negative + score["Negative"]

In [78]:
polarity, positive, negative

(np.float64(3.9999960000040002), np.int64(4), np.int64(0))

# Read the LM Sentiment Words List

In [79]:
lm_positive = pd.read_excel("LoughranMcDonald_SentimentWordLists_2018.xlsx",sheet_name="Positive")
lm_negative = pd.read_excel("LoughranMcDonald_SentimentWordLists_2018.xlsx",sheet_name="Negative")
lm_negative.head()

,ABANDON
0,ABANDONED
1,ABANDONING
2,ABANDONMENT
3,ABANDONMENTS
4,ABANDONS


In [80]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer()

### Vectorize the title snippet

In [81]:
#data['title_snippet']
vectorizer = TfidfVectorizer()
vectors = vectorizer.fit_transform(data['title_snippet'])

In [82]:
matrix = vectors.todense()
dense_list = matrix.tolist()

In [83]:
word_vector = np.array(vectors.mean(axis=0)).flatten().tolist()

In [84]:
# vectors = vectorizer.fit_transform(total_word_list)
# vectors
len(word_vector)

79

### Vectorize the Positve Words of the LM Sentiment Word list

In [85]:
pos_words = [lm_positive.columns[0].lower()] + list(lm_positive.iloc[:, 0].dropna().astype(str).str.lower())
pos_vectors = vectorizer.transform(pos_words)

In [86]:
pos_matrix = pos_vectors.todense()
pos_dense_list = pos_matrix.tolist()

In [87]:
lm_pos_word_vector = np.array(pos_vectors.mean(axis=0)).flatten().tolist()

In [88]:
len(lm_pos_word_vector)

79

### Vectorize the Negative Words of the LM Sentiment Word list

In [89]:
neg_words = [lm_negative.columns[0].lower()] + list(lm_negative.iloc[:, 0].dropna().astype(str).str.lower())
neg_vectors = vectorizer.transform(neg_words)

In [90]:
neg_matrix = neg_vectors.todense()
neg_dense_list = neg_matrix.tolist()

In [91]:
lm_neg_word_vector = np.array(neg_vectors.mean(axis=0)).flatten().tolist()

In [92]:
len(lm_neg_word_vector)

79

### Compute the distance between the title snippet  and the positive words

In [93]:
pos_sqrd_dist = 0
for x in word_vector:
    for y in lm_pos_word_vector:
        diff = x-y
        sqrd_diff = diff*diff
        pos_sqrd_dist = pos_sqrd_dist+sqrd_diff
        

In [94]:
import math
pos_dist = math.sqrt(pos_sqrd_dist)
pos_dist

3.998625911801605

### Compute the distance between the title snippet and the negative words

In [95]:
neg_sqrd_dist = 0
for x in word_vector:    
    for y in lm_neg_word_vector:       
        neg_word_diff = x-y
        neg_sqrd_diff = neg_word_diff*neg_word_diff
        neg_sqrd_dist = neg_sqrd_dist+neg_sqrd_diff

In [96]:
import math
neg_dist = math.sqrt(neg_sqrd_dist)
neg_dist

3.998625911801605

### Compare the distance and extract the sentiment

In [97]:
if(pos_dist < neg_dist):
    print("The Sentiment for the IPO is positive and investors can apply for the IPO")
else:
    print("The Sentiment for the IPO is negative and investors can avoid the IPO")

The Sentiment for the IPO is negative and investors can avoid the IPO


$ \pi $